# SECOND ATTEMPT

### Cyber Attack Detection & Risk Prediction Dataset
Overview
The Cyber Attack Detection & Risk Prediction Dataset is a realistic synthetic dataset containing 100,000+ enterprise cybersecurity incidents. It simulates complete cyber attack lifecycles, from initial access and attacker behavior to incident response, financial impact, and overall risk assessment.
Designed for machine learning, data analysis, and cybersecurity research, this dataset includes realistic relationships between security controls, vulnerabilities, attack progression, and business impact, making it suitable for both academic and industry projects.

In [4]:
# Imports
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import imblearn
import keras
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling, encoding and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier


# Libraries for models
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from xgboost import XGBClassifier


from imblearn.over_sampling import SMOTE

from sklearn import metrics
from sklearn.model_selection import cross_val_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Settings
# pd.set_option('display.max_columns', None)
# import sys
# #np.set_printoptions(threshold=np.nan)
# np.set_printoptions(threshold=sys.maxsize)
# np.set_printoptions(precision=3)
# sns.set(style="darkgrid")
# plt.rcParams['axes.labelsize'] = 14
# plt.rcParams['xtick.labelsize'] = 12
# plt.rcParams['ytick.labelsize'] = 12

In [5]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Load and Preview The Data


In [6]:
# I will be starting with the Enterprise_Cyber_Kill_Chain_Dataset data which is not partitioned yet
ECKCD_dataset = pd.read_csv('/content/drive/MyDrive/solutions/Enterprise_Cyber_Kill_Chain_Dataset.csv')
print(ECKCD_dataset.head())

   Incident_ID            Timestamp    Industry    Country Company_Size  \
0        78303  2025-03-26 15:30:00  Government     Canada        Small   
1        66465  2024-11-23 08:00:00      Retail         UK        Small   
2        93228  2025-08-29 02:45:00   Education  Australia        Small   
3         1748  2023-01-19 04:45:00  Healthcare    Germany        Small   
4        80005  2025-04-13 09:00:00   Education         UK       Medium   

   Employee_Count Attack_Vector    Threat_Actor  Firewall  MFA  ...     Month  \
0             193         Cloud   Script Kiddie         1    0  ...     March   
1             218           Web    Nation State         1    1  ...  November   
2              33         Cloud  Cyber Criminal         1    1  ...    August   
3              84           RDP    Nation State         1    0  ...   January   
4             305           VPN         Insider         1    0  ...     April   

   Business_Hours  Weekend Zero_Day  Compliance  Vendor_Count 

### 2. EDA

In [7]:
# I will then preview the dataset, show its shape and summary statistics
print("The dataset has {} rows and {} columns".format(ECKCD_dataset.shape[0],ECKCD_dataset.shape[1]))
ECKCD_dataset.describe()

The dataset has 100500 rows and 50 columns


,Incident_ID,Employee_Count,Firewall,MFA,EDR,IDS,Security_Training,Patch_Age_Days,Open_Vulnerabilities,CVSS_Score,...,Cyber_Risk_Score,Hour,Business_Hours,Weekend,Zero_Day,Vendor_Count,ThirdParty_Risk,Insider_Risk,Security_Maturity,SOC_Team_Size
count,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,...,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000,100500.000000
mean,50001.764060,2108.605244,0.918249,0.751403,0.799303,0.718746,0.703313,60.591493,4.013433,4.403591,...,45.000796,11.500328,0.416577,0.285055,0.037701,23.297731,34.000402,26.089066,77.820299,9.971761
std,28868.945398,3654.104148,0.273987,0.432202,0.400523,0.449613,0.456799,36.900321,2.352045,1.147416,...,19.049429,6.922520,0.492994,0.451443,0.190474,30.145984,19.732397,20.288059,18.285750,14.602238
min,1.000000,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,1.337935,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,25003.750000,134.000000,1.000000,1.000000,1.000000,0.000000,0.000000,30.000000,2.000000,3.600000,...,30.381794,6.000000,0.000000,0.000000,0.000000,6.000000,21.400000,10.200000,60.000000,2.000000
50%,50004.500000,250.000000,1.000000,1.000000,1.000000,1.000000,1.000000,56.000000,4.000000,4.400000,...,41.632968,11.000000,0.000000,0.000000,0.000000,10.000000,28.400000,16.500000,80.000000,3.000000
75%,75007.250000,1711.000000,1.000000,1.000000,1.000000,1.000000,1.000000,89.000000,5.000000,5.100000,...,57.430332,18.000000,1.000000,1.000000,0.000000,22.000000,38.300000,46.700000,100.000000,8.000000
max,100000.000000,14999.000000,1.000000,1.000000,1.000000,1.000000,1.000000,179.000000,19.000000,10.000000,...,100.000000,23.000000,1.000000,1.000000,1.000000,100.000000,100.000000,93.500000,100.000000,60.000000


In [8]:
print(ECKCD_dataset.columns.tolist())
# From the result, we see that this dataset has so many features that it can be used for both prosspective or retrospective prediction
# I will be going the prospective route, i.e. predicting attacks before an incident occurs
# It has many columns that may not be necessary for the prediction of attacks itself
# Rather, they are dependent on the occurence of attacks such as: Incident_Severity, Incident_ID, Attack_Vector
# It is important to note that in this dataset, each row represents an inciddent (successsful or unsuccessful attempt)
# I will also choosse my Y variable here to be 'Risk_Level' rather than 'Cyber_Risk_Score' for prediction ease and accuracy
# I will then drop the columns I do not need to avoid data leakage

# # identifiers / timing — not posture
#     'Incident_ID', 'Timestamp', 'Hour', 'DayOfWeek', 'Month',
#     'Business_Hours', 'Weekend',

#     # attack characteristics — not knowable before a specific attack happens
#     'Attack_Vector', 'Threat_Actor', 'Attack_Stage', 'Attack_Complexity',
#     'Attack_Success', 'Zero_Day',

#     # attack progression/behavior — only exist once an attack is underway
#     'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation',
#     'Lateral_Movement', 'Persistence', 'Data_Encrypted',

#     # impact/outcome metrics — measured after the incident
#     'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours',
#     'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD',
#     'Data_Exfiltration_GB',

#     # leakage — Risk_Level is derived from these
#     'Cyber_Risk_Score', 'Incident_Severity'

['Incident_ID', 'Timestamp', 'Industry', 'Country', 'Company_Size', 'Employee_Count', 'Attack_Vector', 'Threat_Actor', 'Firewall', 'MFA', 'EDR', 'IDS', 'Security_Training', 'Password_Policy', 'Patch_Age_Days', 'Open_Vulnerabilities', 'CVSS_Score', 'Internet_Exposed', 'Security_Audit_Score', 'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation', 'Lateral_Movement', 'Persistence', 'Data_Encrypted', 'Data_Exfiltration_GB', 'Attack_Success', 'Attack_Stage', 'Attack_Complexity', 'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours', 'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD', 'Cyber_Risk_Score', 'Risk_Level', 'Incident_Severity', 'Hour', 'DayOfWeek', 'Month', 'Business_Hours', 'Weekend', 'Zero_Day', 'Compliance', 'Vendor_Count', 'ThirdParty_Risk', 'Insider_Risk', 'Security_Maturity', 'SOC_Team_Size']


##### Initial Feature Dropping

In [9]:
# specify features to be droppedd and drop them

# features_to_drop = ['Incident_ID', 'Timestamp', 'Hour', 'DayOfWeek', 'Month',
#     'Business_Hours', 'Weekend', 'Attack_Vector', 'Threat_Actor', 'Attack_Stage', 'Attack_Complexity',
#     'Attack_Success', 'Zero_Day', 'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation',
#     'Lateral_Movement', 'Persistence', 'Data_Encrypted',     'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours',
#     'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD',
#     'Data_Exfiltration_GB', 'Cyber_Risk_Score', 'Incident_Severity']

features_to_drop = ['Incident_ID', 'Attack_Vector', 'Threat_Actor', 'Attack_Stage', 'Attack_Complexity',
    'Attack_Success', 'Zero_Day', 'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation',
    'Lateral_Movement', 'Persistence', 'Data_Encrypted',  'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours',
    'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD',
    'Data_Exfiltration_GB', 'Cyber_Risk_Score', 'Incident_Severity']

data = ECKCD_dataset.drop(columns=features_to_drop)



In [10]:
# Investigate Class Imbalance
print(data['Risk_Level'].value_counts())
# See it in percentage
print(data['Risk_Level'].value_counts(normalize=True)*100)

Risk_Level
Medium      51262
High        27104
Low         13741
Critical     8393
Name: count, dtype: int64
Risk_Level
Medium      51.006965
High        26.969154
Low         13.672637
Critical     8.351244
Name: proportion, dtype: float64


### 3. Data Cleaning - Missing, Infinite and Duplicate Values

In [11]:
# Null Values
null_values = data.isnull().sum()
print(null_values[null_values>0])
# we see there are 42,983 missing values in this dataset, all belonging to the Compliance column
data['Compliance'] = data['Compliance'].fillna('None')

# Infinite Values
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
inf_values = np.isinf(data[numeric_cols]).sum()
print(inf_values)
# No infinite values

# Duplicate Values
duplicate_values = data.duplicated().sum()
print(duplicate_values) # They were 500, I will drop them below
data = data.drop_duplicates()

Compliance    42983
dtype: int64
Employee_Count          0
Firewall                0
MFA                     0
EDR                     0
IDS                     0
Security_Training       0
Patch_Age_Days          0
Open_Vulnerabilities    0
CVSS_Score              0
Internet_Exposed        0
Security_Audit_Score    0
Hour                    0
Business_Hours          0
Weekend                 0
Vendor_Count            0
ThirdParty_Risk         0
Insider_Risk            0
Security_Maturity       0
SOC_Team_Size           0
dtype: int64
500


### 4. Encoding Categorical Columns in Independent Variables (X)

In [12]:
# Define independent and dependent variables
x = data.drop(columns=['Risk_Level'])
y = data['Risk_Level']


# define non-numeric columns
cat_cols = x.select_dtypes(include=['object']).columns
print(cat_cols)

# Encode categorical variables
x = pd.get_dummies(x, columns=cat_cols, drop_first=True)
print(x.shape)

Index(['Timestamp', 'Industry', 'Country', 'Company_Size', 'Password_Policy',
       'DayOfWeek', 'Month', 'Compliance'],
      dtype='object')
(100000, 100057)


### 5. Train/Test Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.8, random_state=7, stratify=y
)

# Addded this afterwardd to handle imbalalance
smote = SMOTE(random_state=7)
x_train, y_train = smote.fit_resample(x_train, y_train)

### 6. Scale Numerical Data

In [ ]:
scaler = StandardScaler()
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
dummy_cols = [c for c in x.columns if c not in num_cols]

sc_train = scaler.fit_transform(x_train[num_cols])
sc_test = scaler.transform(x_test[num_cols])

sc_traindf = pd.DataFrame(sc_train, columns=num_cols, index=x_train.index)
sc_testdf = pd.DataFrame(sc_test, columns=num_cols, index=x_test.index)

### 7. Encode Dependent Variable (Y)

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [ ]:
# I am reassigning dataframes to variables with simpler names for ease

X = pd.concat([sc_traindf, x_train[dummy_cols].astype(int)], axis=1)
Y = y_train_encoded.copy()
X_TEST = pd.concat([sc_testdf, x_test[dummy_cols].astype(int)], axis=1)
Y_TEST = y_test_encoded.copy()

### 8. Feature Selection

In [ ]:
# Define and train the feature classifier
rfc = RandomForestClassifier(random_state=7)
rfc.fit(X, Y)

# Create a dataframe showing features against their importance score
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': np.round(rfc.feature_importances_, 3)
}).sort_values('importance', ascending=False)

# Plot a bar chart for visualisation
importances.head(25).plot.bar(figsize=(12, 4), x = 'feature', y='importance')
plt.show()

# Feature only the top features in X (train and Test)
top_features = importances.head(25)['feature'].tolist()
X = X[top_features]
X_TEST = X_TEST[top_features]

### 9. Train ML Models

In [ ]:
KNN_Classifier = KNeighborsClassifier(n_jobs=-1)
KNN_Classifier.fit(X, Y)

LGR_Classifier = LogisticRegression(n_jobs=-1, random_state=7, max_iter=1000, class_weight='balanced')
LGR_Classifier.fit(X, Y)

BNB_Classifier = BernoulliNB()
BNB_Classifier.fit(X, Y)

DTC_Classifier = tree.DecisionTreeClassifier(criterion='entropy', random_state=7, class_weight='balanced', max_depth=10, min_samples_leaf=20)
DTC_Classifier.fit(X, Y)

XGB_Classifier = XGBClassifier(random_state=7, n_estimators=200, max_depth=8, learning_rate=0.1, objective="multi:softmax", eval_metric="mlogloss")
XGB_Classifier.fit(X, Y)

### 10. Evaluate Model (on training data)

In [ ]:
models = []
models.append(('Naive Bayes', BNB_Classifier))
models.append(('Decision Tree', DTC_Classifier))
models.append(('KNN', KNN_Classifier))
models.append(('Logistic Regression', LGR_Classifier))
models.append(("XGBoost", XGB_Classifier))

for name, model in models:
    cv_scores = cross_val_score(model, X, Y, cv=10)
    accuracy = metrics.accuracy_score(Y, model.predict(X))
    conf_matrix = metrics.confusion_matrix(Y, model.predict(X))
    report = metrics.classification_report(Y, model.predict(X))

    print(f"\n===== {name} - Train Evaluation =====")
    print("Cross Validation Mean Score:", cv_scores.mean())
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)

### Valiate Model (on test data)

In [ ]:

for name, model in models:
    accuracy = metrics.accuracy_score(Y_TEST, model.predict(X_TEST))
    conf_matrix = metrics.confusion_matrix(Y_TEST, model.predict(X_TEST))
    report = metrics.classification_report(Y_TEST, model.predict(X_TEST))

    print(f"\n===== {name} - Validation Results =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)

### Train and Evaluate DL Model (ANN)

In [ ]:
n_classes = len(le.classes_)
Y_cat = to_categorical(Y, num_classes=n_classes)
Y_TEST_cat = to_categorical(Y_TEST, num_classes=n_classes)

model = Sequential([
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model.fit(
    X, Y_cat,
    epochs=20, batch_size=32,
    validation_data=(X_TEST, Y_TEST_cat),  # real holdout, not a SMOTE'd split
    callbacks=[early_stop]
)

train_pred_probs = model.predict(X)
train_pred_classes = np.argmax(train_pred_probs, axis=1)

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred_classes))
print(metrics.classification_report(Y, train_pred_classes))

pred_probs = model.predict(X_TEST)
pred_classes = np.argmax(pred_probs, axis=1)

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, pred_classes))
print(metrics.classification_report(Y_TEST, pred_classes))

## Code Before Early Stopping and SMOTE
# n_classes = len(le.classes_)
# Y_cat = to_categorical(Y, num_classes=n_classes)
# Y_TEST_cat = to_categorical(Y_TEST, num_classes=n_classes)

# # model = Sequential()
# # model.add(Dense(64, activation='relu', input_shape=(X.shape[1],)))
# # model.add(Dense(32, activation='relu'))
# # model.add(Dense(n_classes, activation='softmax'))

# ## Make DL architecture deeper
# model = Sequential([
#     Dense(256, activation='relu'),
#     Dense(128, activation='relu'),
#     Dense(64, activation='relu'),
#     Dense(n_classes, activation='softmax')
# ])

# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# model.fit(X, Y_cat, epochs=20, batch_size=32, validation_split=0.1)

# train_pred_probs = model.predict(X)
# train_pred_classes = np.argmax(train_pred_probs, axis=1)

# print("Train Accuracy:", metrics.accuracy_score(Y, train_pred_classes))
# print(metrics.classification_report(Y, train_pred_classes))

# pred_probs = model.predict(X_TEST)
# pred_classes = np.argmax(pred_probs, axis=1)

# print("Accuracy:", metrics.accuracy_score(Y_TEST, pred_classes))
# print(metrics.classification_report(Y_TEST, pred_classes))